## Valor presente de las cuotas pendientes

Partimos del dataframe que sale del notebook `amortizacion`, que ya trae la columna `Sistema` (alemán o francés) para cada cliente.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("Data/df_negocio_amortizacion.csv")
df[["ID", "Monto_Inicial", "Ratio_Interes", "Duracion", "Sistema"]].head()

### Tablas de amortización

Hacen falta para obtener las cuotas de cada préstamo.

In [ ]:
def amortizacion_alemana(capital, interes_anual, plazo_meses):
    r = interes_anual / 100 / 12
    A = capital / plazo_meses
    
    tabla = []
    saldo = capital
    
    for t in range(1, plazo_meses + 1):
        interes = saldo * r
        cuota = A + interes
        saldo -= A
        
        tabla.append({
            "Mes": t,
            "Amortizacion": round(A, 2),
            "Interes": round(interes, 2),
            "Cuota": round(cuota, 2),
            "Saldo_restante": round(max(saldo, 0), 2)
        })
    
    return pd.DataFrame(tabla)

def reduccion_mensual(capital, interes_anual, plazo_meses):
    r = interes_anual / 100 / 12      
    A = capital / plazo_meses
    return round(r * A, 2)


In [ ]:
def amortizacion_francesa(capital, interes_anual, plazo_meses):
    i = interes_anual / 100 / 12
    cuota = capital * i / (1 - (1 + i) ** (-plazo_meses))

    tabla = []
    saldo = capital

    for t in range(1, plazo_meses + 1):
        interes = saldo * i
        amortizacion = cuota - interes
        saldo -= amortizacion

        tabla.append({
            "Mes": t,
            "Amortizacion": round(amortizacion, 2),
            "Interes": round(interes, 2),
            "Cuota": round(cuota, 2),
            "Saldo_restante": round(max(saldo, 0), 2)
        })

    return pd.DataFrame(tabla)

### Cálculo del valor presente

In [ ]:
def valor_renta(i, momento="vencida", n=None, cuota=None, flujos=None, g=None, d=None):
    aj = (1 + i) if momento == "anticipada" else 1

    if flujos is not None:                                   # variable temporal
        flujos = np.asarray(flujos, dtype=float)
        n = len(flujos)
        k = np.arange(1, n + 1) - (1 if momento == "anticipada" else 0)
        va = (flujos / (1 + i) ** k).sum()
        return va, va * (1 + i) ** n

    if n is None:                                            # perpetuas
        if g is not None:
            va = cuota / (i - g) * aj
        elif d is not None:
            va = (cuota / i + d / i ** 2) * aj
        else:
            va = cuota / i * aj
        return va, np.inf                                    # el valor final no existe

    va = cuota * (1 - (1 + i) ** -n) / i * aj                # constante temporal
    return va, va * (1 + i) ** n

In [ ]:
def valor_presente_cuotas_pendientes(capital, tin_pct, plazo_meses, sistema):
    tabla = amortizacion_alemana(capital, tin_pct, plazo_meses) if sistema == "aleman" \
        else amortizacion_francesa (capital, tin_pct, plazo_meses)

    mitad = plazo_meses // 2
    cuotas_pendientes = tabla[tabla["Mes"] > mitad]["Cuota"]

    i = tin_pct / 100 / 12
    va, vf = valor_renta(i, "vencida", flujos=cuotas_pendientes)

    return round(va, 2), round(vf, 2)

In [ ]:
def vp_fila(f):
    if f[["Monto_Inicial", "Ratio_Interes", "Duracion"]].isna().any():
        return (np.nan, np.nan)

    return valor_presente_cuotas_pendientes(
        f["Monto_Inicial"],
        f["Ratio_Interes"],
        int(f["Duracion"]),
        f["Sistema"]
    )

df[["Valor_Presente", "Valor_Final"]] = df.apply(
    vp_fila,
    axis=1,
    result_type="expand"
)

In [ ]:
df[["ID", "Sistema", "Monto_Inicial", "Duracion", "Valor_Presente", "Valor_Final"]].head(10)

In [ ]:
df.to_csv("Data/df_negocio_valor_presente.csv", index=False)